# Tool
tính khoảng cách giữa hai tọa độ,
 ước lượng quãng đường đường bộ,
 tính thời gian di chuyển theo phương tiện,
 lấy tuyến đường từ OSRM,
 chọn chi nhánh tối ưu theo vị trí giao hàng

In [1]:
import math

try:
    import requests
except ImportError:
    requests = None

TRAFFIC_CONFIG = {
    "average_speed": {
        "motorbike": 35,
        "car": 30,
        "walking": 4.5,
    },
    "road_distance_factor": {
        "motorbike": 1.2,
        "car": 1.2,
        "walking": 1.2,
    },
}

OSRM_BASE_URL = "http://router.project-osrm.org/route/v1"


In [2]:
# Hàm 1 - Tính khoảng cách đường chim bay
def calculate_air_distance_km(start_lat, start_lng, end_lat, end_lng):
    """
    Tính khoảng cách đường chim bay giữa 2 tọa độ bằng công thức Haversine.
    """
    earth_radius_km = 6371.0

    start_lat = float(start_lat)
    start_lng = float(start_lng)
    end_lat = float(end_lat)
    end_lng = float(end_lng)

    delta_lat = math.radians(end_lat - start_lat)
    delta_lng = math.radians(end_lng - start_lng)

    a = (
        math.sin(delta_lat / 2) ** 2
        + math.cos(math.radians(start_lat))
        * math.cos(math.radians(end_lat))
        * math.sin(delta_lng / 2) ** 2
    )
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return earth_radius_km * c


In [3]:
# Hàm 2 - Ước lượng quãng đường đi thực tế
def estimate_road_distance_km(air_distance_km, delivery_mode="motorbike"):
    """
    Ước lượng quãng đường di chuyển thực tế từ khoảng cách đường chim bay.
    Giữ cùng một quãng đường cơ sở cho mọi phương tiện; chỉ khác thời gian di chuyển.
    """
    road_factor = TRAFFIC_CONFIG["road_distance_factor"].get("motorbike", 1.2)
    return air_distance_km * road_factor


In [4]:
# Hàm 3 - Tính thời gian di chuyển dự kiến
def estimate_travel_time_minutes(estimated_distance_km, delivery_mode="motorbike"):
    """
    Tính thời gian di chuyển dự kiến theo quãng đường và vận tốc trung bình.
    """
    average_speed = TRAFFIC_CONFIG["average_speed"].get(delivery_mode, 30)

    if average_speed <= 0:
        return 1

    estimated_minutes = int(round((estimated_distance_km / average_speed) * 60))
    return max(estimated_minutes, 1)


In [5]:
# Hàm 4 - Gọi OSRM để lấy tuyến đường thực tế
def get_osrm_route_data(start_lat, start_lng, end_lat, end_lng):
    """
    Gọi OSRM một lần để lấy hình học tuyến và quãng đường cơ sở dùng chung cho mọi phương tiện.
    Nếu không gọi được API thì trả về tuyến fallback nối thẳng 2 điểm.
    """
    fallback_points = [
        [float(start_lat), float(start_lng)],
        [float(end_lat), float(end_lng)],
    ]

    if requests is None:
        return {
            "route_points": fallback_points,
            "distance_km": None,
        }

    route_coordinates = f"{start_lng},{start_lat};{end_lng},{end_lat}"
    request_url = f"{OSRM_BASE_URL}/driving/{route_coordinates}"
    query_params = {
        "overview": "full",
        "geometries": "geojson",
        "steps": "true",
        "alternatives": "false",
    }
    request_headers = {
        "User-Agent": "Mozilla/5.0",
    }

    try:
        response = requests.get(
            request_url,
            params=query_params,
            headers=request_headers,
            timeout=10,
        )
        response_data = response.json()
    except Exception:
        return {
            "route_points": fallback_points,
            "distance_km": None,
        }

    if response.status_code == 200 and response_data.get("code") == "Ok":
        route_list = response_data.get("routes", [])
        if route_list:
            route_item = route_list[0]
            geometry = route_item.get("geometry") or {}
            route_coordinates = geometry.get("coordinates", [])
            route_points = [
                [point[1], point[0]]
                for point in route_coordinates
                if len(point) >= 2
            ] or fallback_points
            distance_km = None
            try:
                distance_km = round(float(route_item.get("distance", 0)) / 1000, 2)
            except (TypeError, ValueError):
                distance_km = None
            return {
                "route_points": route_points,
                "distance_km": distance_km,
            }

    return {
        "route_points": fallback_points,
        "distance_km": None,
    }


In [6]:
# Hàm 5 - Ghép các bước GIS để tạo kết quả tuyến hoàn chỉnh
def estimate_route_gis(start_lat, start_lng, end_lat, end_lng, delivery_mode="motorbike"):
    """
    Ước lượng tuyến giao hàng giữa 2 điểm.
    Chỉ giữ phần tính toán GIS, không tính phí ship trong notebook demo này.
    """
    try:
        start_lat = float(start_lat)
        start_lng = float(start_lng)
        end_lat = float(end_lat)
        end_lng = float(end_lng)
    except (TypeError, ValueError):
        return {"error": "Tọa độ lỗi."}

    if delivery_mode not in TRAFFIC_CONFIG["average_speed"]:
        delivery_mode = "motorbike"

    air_distance_km = calculate_air_distance_km(start_lat, start_lng, end_lat, end_lng)
    estimated_distance_km = estimate_road_distance_km(air_distance_km, delivery_mode)

    route_data = get_osrm_route_data(
        start_lat,
        start_lng,
        end_lat,
        end_lng,
    )

    route_points = route_data.get("route_points") or [
        [start_lat, start_lng],
        [end_lat, end_lng],
    ]
    shared_distance_km = route_data.get("distance_km")
    if shared_distance_km is None:
        shared_distance_km = round(estimated_distance_km, 2)

    estimated_duration_min = estimate_travel_time_minutes(shared_distance_km, delivery_mode)

    return {
        "distance_km": round(shared_distance_km, 2),
        "duration_min": estimated_duration_min,
        "route_points": route_points,
        "mode": delivery_mode,
        "air_distance_km": round(air_distance_km, 2),
    }


In [7]:
# Hàm 6 - Chọn nhà thuốc tối ưu theo khoảng cách
def choose_best_pharmacy_fast_gis(pharmacies, delivery_lat, delivery_lng, delivery_mode="motorbike"):
    """
    Chọn chi nhánh phù hợp nhất cho vị trí giao hàng.
    Cách chọn: duyệt từng chi nhánh, so sánh quãng đường ước lượng, lấy chi nhánh gần nhất.
    Sau đó gọi lại estimate_route_gis để lấy tuyến chi tiết.
    """
    if not pharmacies:
        return {"error": "Không có nhà thuốc khả dụng."}

    best_pharmacy = None
    shortest_distance_km = None

    for pharmacy in pharmacies:
        air_distance_km = calculate_air_distance_km(
            pharmacy["lat"],
            pharmacy["lng"],
            delivery_lat,
            delivery_lng,
        )
        current_distance_km = estimate_road_distance_km(air_distance_km, delivery_mode)

        if shortest_distance_km is None or current_distance_km < shortest_distance_km:
            shortest_distance_km = current_distance_km
            best_pharmacy = pharmacy

    if best_pharmacy is None:
        return {"error": "Không tìm được chi nhánh phù hợp."}

    route_result = estimate_route_gis(
        start_lat=best_pharmacy["lat"],
        start_lng=best_pharmacy["lng"],
        end_lat=delivery_lat,
        end_lng=delivery_lng,
        delivery_mode=delivery_mode,
    )

    return {
        "pharmacy": best_pharmacy,
        "route": route_result,
    }


In [8]:
# Dữ liệu đầu vào demo
pharmacies = [
    {"id": 9101, "name": "Pharmacy - Quận 1", "lat": 10.763782, "lng": 106.696376},
    {"id": 9102, "name": "Pharmacy - Quận 3", "lat": 10.786152, "lng": 106.678208},
    {"id": 9112, "name": "City Drugstore - Quận 5", "lat": 10.758497, "lng": 106.677987},
]

delivery_lat = 10.775000
delivery_lng = 106.700000
delivery_mode = "motorbike"

print("Điểm giao hàng mẫu:")
print({"lat": delivery_lat, "lng": delivery_lng, "mode": delivery_mode})


Điểm giao hàng mẫu:
{'lat': 10.775, 'lng': 106.7, 'mode': 'motorbike'}


In [9]:
# Demo từng bước cho từng nhà thuốc
for pharmacy in pharmacies:
    air_distance = calculate_air_distance_km(
        pharmacy["lat"],
        pharmacy["lng"],
        delivery_lat,
        delivery_lng,
    )
    road_distance = estimate_road_distance_km(air_distance, delivery_mode)
    duration_min = estimate_travel_time_minutes(road_distance, delivery_mode)

    print("-" * 60)
    print("Nhà thuốc:", pharmacy["name"])
    print("Khoảng cách chim bay (km):", round(air_distance, 2))
    print("Quãng đường ước lượng (km):", round(road_distance, 2))
    print("Thời gian di chuyển dự kiến (phút):", duration_min)


------------------------------------------------------------
Nhà thuốc: Pharmacy - Quận 1
Khoảng cách chim bay (km): 1.31
Quãng đường ước lượng (km): 1.57
Thời gian di chuyển dự kiến (phút): 3
------------------------------------------------------------
Nhà thuốc: Pharmacy - Quận 3
Khoảng cách chim bay (km): 2.68
Quãng đường ước lượng (km): 3.22
Thời gian di chuyển dự kiến (phút): 6
------------------------------------------------------------
Nhà thuốc: City Drugstore - Quận 5
Khoảng cách chim bay (km): 3.02
Quãng đường ước lượng (km): 3.63
Thời gian di chuyển dự kiến (phút): 6


In [10]:
# Demo lấy tuyến chi tiết cho một nhà thuốc
route_demo = estimate_route_gis(
    start_lat=pharmacies[0]["lat"],
    start_lng=pharmacies[0]["lng"],
    end_lat=delivery_lat,
    end_lng=delivery_lng,
    delivery_mode=delivery_mode,
)

print("Kết quả estimate_route_gis:")
print("distance_km =", route_demo["distance_km"])
print("duration_min =", route_demo["duration_min"])
print("mode =", route_demo["mode"])
print("Số điểm trên tuyến =", len(route_demo["route_points"]))
print("5 điểm đầu tuyến =", route_demo["route_points"][:5])


Kết quả estimate_route_gis:
distance_km = 1.94
duration_min = 3
mode = motorbike
Số điểm trên tuyến = 72
5 điểm đầu tuyến = [[10.7641, 106.696059], [10.764172, 106.696134], [10.764275, 106.69625], [10.764385, 106.696377], [10.764429, 106.696434]]


In [11]:
# Demo kết quả cuối cùng: chọn nhà thuốc tối ưu
best_result = choose_best_pharmacy_fast_gis(
    pharmacies=pharmacies,
    delivery_lat=delivery_lat,
    delivery_lng=delivery_lng,
    delivery_mode=delivery_mode,
)

print("Chi nhánh tối ưu:")
print(best_result["pharmacy"])
print("Tuyến giao hàng:")
print(best_result["route"])


Chi nhánh tối ưu:
{'id': 9101, 'name': 'Pharmacy - Quận 1', 'lat': 10.763782, 'lng': 106.696376}
Tuyến giao hàng:
{'distance_km': 1.94, 'duration_min': 3, 'route_points': [[10.7641, 106.696059], [10.764172, 106.696134], [10.764275, 106.69625], [10.764385, 106.696377], [10.764429, 106.696434], [10.764576, 106.696602], [10.764673, 106.696719], [10.764753, 106.696835], [10.764775, 106.696865], [10.764937, 106.697085], [10.764844, 106.69714], [10.76447, 106.697361], [10.764172, 106.697534], [10.764301, 106.697743], [10.764569, 106.69759], [10.764601, 106.697571], [10.764625, 106.697558], [10.765094, 106.697303], [10.765169, 106.697247], [10.76524, 106.697132], [10.765882, 106.696713], [10.766374, 106.696426], [10.766964, 106.696082], [10.767048, 106.696035], [10.767117, 106.695993], [10.767353, 106.695854], [10.767918, 106.695533], [10.768007, 106.695521], [10.768081, 106.695488], [10.768111, 106.69551], [10.768126, 106.695522], [10.768321, 106.695694], [10.768545, 106.695893], [10.76893, 